# PhytoLabs on Google Colab — wheat brown-rust pipeline

Runs the full two-stage pipeline (GMM segmentation -> from-scratch logistic regression) in Colab, where the local macOS `psutil` kernel crash does not apply.

**Steps:** clone repo -> install -> (Kaggle) download data -> reshape -> train GMM -> features -> logistic regression -> visualize.

Fill in `REPO_URL` and, when you have it, `KAGGLE_DATASET` below.

## 1. Clone the repo and install

In [ ]:
# TODO: replace with your GitHub repo URL after pushing.
REPO_URL = "https://github.com/<your-username>/phytolabs.git"

import os
if not os.path.isdir("phytolabs"):
    !git clone $REPO_URL
%cd phytolabs
!pip -q install -e .

## 2. (Optional) Smoke test with synthetic data
Confirms the whole pipeline runs in Colab before touching real data.

In [ ]:
!python -m phytolabs.cli make-synthetic --data-dir data
!python -m phytolabs.cli train-gmm --data-dir data --k 4
!python -m phytolabs.cli build-features --data-dir data
!python -m phytolabs.cli train-logreg

## 3. Real Kaggle dataset
Upload your `kaggle.json` API token (Kaggle -> Account -> Create New Token), then download and unzip the dataset.

Set `KAGGLE_DATASET` to the dataset slug (e.g. `olyadgetch/wheat-leaf-dataset` or `shadabhussain/cgiar-computer-vision-for-crop-disease`).

In [ ]:
from google.colab import files
print('Upload your kaggle.json:')
files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle

In [ ]:
# TODO: set the dataset slug once you pick it.
KAGGLE_DATASET = "olyadgetch/wheat-leaf-dataset"

!rm -rf data/raw && mkdir -p data/raw
!kaggle datasets download -d $KAGGLE_DATASET -p data/raw --unzip
!find data/raw -maxdepth 2 -type d | sort

Inspect the printed folders above, then point the reshape helper at the **healthy** and **brown/leaf rust** class folders to build `data/{train,val}/{healthy,rust}/`.

In [ ]:
# TODO: edit these two paths to match the printed folder structure.
HEALTHY_SRC = "data/raw/Healthy"
RUST_SRC = "data/raw/leaf_rust"

# Start clean so synthetic smoke-test images don't mix with real data.
!rm -rf data/train data/val
!python -m scripts.reshape_data --healthy-src "$HEALTHY_SRC" --rust-src "$RUST_SRC" --out data --val-fraction 0.2
!echo 'train/healthy:' $(ls data/train/healthy | wc -l) ' train/rust:' $(ls data/train/rust | wc -l)

## 4. Run the pipeline on the real data (in-process, with inline visuals)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import io, segmentation, pipeline, viz, metrics, calibration
from phytolabs.features import FEATURE_NAMES
from phytolabs.logreg import LogisticRegressionSGD

DATA = 'data'

# Stage 1: fit the GMM on the training images.
leaf_gmm = pipeline.fit_gmm_from_dir(f'{DATA}/train', k=4)
for c in range(leaf_gmm.n_components):
    h, s, v = leaf_gmm.gmm.means_[c]
    print(f'component {c}: HSV mean=({h:.0f},{s:.0f},{v:.0f}) -> {leaf_gmm.label_map[c]}')

In [ ]:
# Stage 1 -> 2: build feature tables.
X_tr, y_tr, _ = pipeline.build_feature_table(f'{DATA}/train', leaf_gmm)
X_va, y_va, _ = pipeline.build_feature_table(f'{DATA}/val', leaf_gmm)
print('train', X_tr.shape, 'val', X_va.shape)
viz.plot_feature_histograms(X_tr, y_tr, FEATURE_NAMES); plt.show()

In [ ]:
# Stage 2: from-scratch logistic regression + SGD.
model = LogisticRegressionSGD(lr=0.1, epochs=300, batch_size=16, l2=1e-3).fit(X_tr, y_tr)
proba = model.predict_proba(X_va)
pred = model.predict(X_va)
print(metrics.classification_report(y_va, pred, proba))
print('band:', calibration.band_summary(proba, 0.45, 0.55))
viz.plot_loss(model.loss_history); plt.show()
viz.plot_confusion_matrix(y_va, pred); plt.show()
viz.plot_roc(y_va, proba); plt.show()
viz.plot_reliability(y_va, proba, n_bins=5); plt.show()

In [ ]:
# Lesion-overlay gallery (the product UX) on a few validation images.
samples = []
for cls in ('rust', 'healthy'):
    for p in list(io.iter_image_paths(f'{DATA}/val/{cls}'))[:4]:
        result, seg, bgr = pipeline.predict_image(p, leaf_gmm, model)
        samples.append((bgr, seg['rust'], f"{cls}: P={result['probability']:.2f} ({result['label']})"))
viz.overlay_gallery(samples, ncols=4); plt.show()